# 🔍 Financial Fraud Detection Model
### Project 1 | Data Science Project | Rohit

---

## What This Notebook Does
This notebook builds a complete **Financial Fraud Detection System** step by step:

1. **Load & Explore** the dataset (understand what data we have)
2. **Clean & Preprocess** the data (prepare it for ML)
3. **Handle Class Imbalance** using SMOTE (fraud is rare — we fix that)
4. **Train 6 Machine Learning Models** (from simple to advanced)
5. **Evaluate & Compare** all models (find the best one)
6. **Save the Best Model** (to use in our dashboard later)

> 💡 **New to ML?** Every cell has a comment explaining *what* it does and *why*.


---
## 📦 Section 1 — Install & Import Libraries
*We bring in all the tools we'll need. Run this cell first.*

In [ ]:
# ── Install all required libraries (auto-installs, ~1 min first run) ─────
import subprocess, sys

packages = [
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'scikit-learn', 'imbalanced-learn', 'xgboost', 'joblib'
]

print('Installing packages... (takes ~1-2 minutes the first time)')
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages installed!')
print()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, IsolationForest
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)

import joblib, warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully!')


---
## 📂 Section 2 — Load the Dataset
We use `synthetic_fraud_dataset1.csv` — 50,000 transactions with a Fraud label (0 = legitimate, 1 = fraud).


In [ ]:
# Load the primary dataset
# (Make sure this notebook is in the Project_1_Fraud_Detection/ folder,
#  and the CSV is inside the data/ subfolder)
df = pd.read_csv('data/synthetic_fraud_dataset1.csv')

print(f"✅ Dataset loaded!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print()
print("Column names:")
print(df.columns.tolist())


In [ ]:
# Preview the first 5 rows — like opening Excel and scrolling to the top
df.head()


In [ ]:
# Data types and non-null counts for each column
# This tells us which columns are numbers vs text, and if any data is missing
df.info()


In [ ]:
# Statistical summary: mean, min, max, etc. for numeric columns
df.describe()


---
## 📊 Section 3 — Exploratory Data Analysis (EDA)
**EDA = Understanding your data before modelling.**
We look at distributions, patterns, and the fraud rate.


In [ ]:
# ── 3.1  Check for missing values ────────────────────────────────────────────
# Missing values can break ML models — we check first
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")


In [ ]:
# ── 3.2  Check for duplicate rows ────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"Removed {dupes} duplicate rows.")


In [ ]:
# ── 3.3  Fraud vs Legitimate — Class Distribution ────────────────────────────
# This is the most important EDA step for fraud detection.
# "Class imbalance" = fraud cases are much rarer than legitimate ones.

fraud_counts = df['Fraud_Label'].value_counts()
fraud_pct    = df['Fraud_Label'].value_counts(normalize=True) * 100

print("Transaction counts:")
print(f"  Legitimate (0): {fraud_counts[0]:,}  ({fraud_pct[0]:.1f}%)")
print(f"  Fraud      (1): {fraud_counts[1]:,}  ({fraud_pct[1]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values,
            color=['#2196F3', '#F44336'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Transaction Count by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            colors=['#2196F3', '#F44336'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Chart saved as eda_class_distribution.png")


In [ ]:
# ── 3.4  Transaction Amount Distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of all amounts
axes[0].hist(df['Transaction_Amount'], bins=50, color='#2196F3', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')

# Fraud vs Legitimate amounts side by side
df[df['Fraud_Label']==0]['Transaction_Amount'].hist(
    bins=40, alpha=0.6, color='#2196F3', label='Legitimate', ax=axes[1])
df[df['Fraud_Label']==1]['Transaction_Amount'].hist(
    bins=40, alpha=0.6, color='#F44336', label='Fraud', ax=axes[1])
axes[1].set_title('Amount: Fraud vs Legitimate', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_transaction_amount.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.5  Fraud rate by Transaction Type ──────────────────────────────────────
fraud_by_type = (df.groupby('Transaction_Type')['Fraud_Label']
                   .mean()
                   .sort_values(ascending=False) * 100)

plt.figure(figsize=(10, 4))
bars = plt.bar(fraud_by_type.index, fraud_by_type.values,
               color=['#F44336' if v > fraud_by_type.mean() else '#2196F3'
                      for v in fraud_by_type.values])
plt.title('Fraud Rate (%) by Transaction Type', fontsize=13, fontweight='bold')
plt.xlabel('Transaction Type')
plt.ylabel('Fraud Rate (%)')
for bar, val in zip(bars, fraud_by_type.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fraud_by_type.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.6  Fraud rate by Device Type & Location ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# By Device Type
fraud_device = (df.groupby('Device_Type')['Fraud_Label'].mean() * 100).sort_values(ascending=False)
axes[0].bar(fraud_device.index, fraud_device.values, color='#FF7043')
axes[0].set_title('Fraud Rate (%) by Device Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
for i, v in enumerate(fraud_device.values):
    axes[0].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9)

# By Top 10 Locations
fraud_loc = (df.groupby('Location')['Fraud_Label'].mean() * 100).sort_values(ascending=False).head(10)
axes[1].barh(fraud_loc.index[::-1], fraud_loc.values[::-1], color='#7E57C2')
axes[1].set_title('Fraud Rate (%) — Top 10 Locations', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fraud Rate (%)')

plt.tight_layout()
plt.savefig('eda_fraud_by_device_location.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.7  Boxplots — Outlier Detection ────────────────────────────────────────
# A boxplot shows the spread of data. Dots outside the "whiskers" = outliers.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.boxplot(column='Transaction_Amount', by='Fraud_Label', ax=axes[0])
axes[0].set_title('Transaction Amount by Fraud Label')
axes[0].set_xlabel('0 = Legitimate, 1 = Fraud')

df.boxplot(column='Account_Balance', by='Fraud_Label', ax=axes[1])
axes[1].set_title('Account Balance by Fraud Label')
axes[1].set_xlabel('0 = Legitimate, 1 = Fraud')

plt.suptitle('')
plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 🔧 Section 4 — Data Preprocessing
**Preprocessing = Transforming raw data into a form that ML models can understand.**

Steps we'll take:
1. Remove columns the model doesn't need (IDs, raw dates)
2. Encode text columns → numbers (ML models only understand numbers)
3. Remove outliers (extreme values that skew the model)
4. Scale numeric features (put everything on the same scale)


In [ ]:
# ── 4.1  Make a working copy — never modify the original ────────────────────
df_clean = df.copy()

# Columns to drop: IDs and raw date (not useful as features)
df_clean = df_clean.drop(columns=['Transaction_ID', 'User_ID', 'Date'])

print("Remaining columns:")
print(df_clean.columns.tolist())
print(f"Shape: {df_clean.shape}")


In [ ]:
# ── 4.2  Label Encode all categorical (text) columns ─────────────────────────
# LabelEncoder converts text → numbers, e.g. "Online" → 0, "POS" → 2
# We store each encoder so we can reverse the transformation later if needed.

categorical_cols = ['Transaction_Type', 'Device_Type', 'Location',
                    'Merchant_Category', 'Card_Type']

label_encoders = {}   # dictionary to store each encoder

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f"✅ Encoded '{col}' — unique values: {le.classes_.tolist()}")


In [ ]:
# ── 4.3  Outlier Removal using IQR Method ───────────────────────────────────
# IQR (Interquartile Range): the middle 50% of data.
# Anything beyond 1.5× IQR outside Q1/Q3 is considered an outlier.

def remove_outliers_iqr(dataframe, column):
    Q1  = dataframe[column].quantile(0.25)
    Q3  = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(dataframe)
    dataframe = dataframe[(dataframe[column] >= lower) & (dataframe[column] <= upper)]
    removed = before - len(dataframe)
    print(f"'{column}': removed {removed:,} outliers (lower={lower:.2f}, upper={upper:.2f})")
    return dataframe

df_clean = remove_outliers_iqr(df_clean, 'Transaction_Amount')
df_clean = remove_outliers_iqr(df_clean, 'Account_Balance')

print(f"\nDataset size after outlier removal: {df_clean.shape[0]:,} rows")


In [ ]:
# ── 4.4  Define Features (X) and Target (y) ──────────────────────────────────
# X = all the columns the model learns from (inputs / features)
# y = what we want to predict (Fraud_Label: 0 or 1)

X = df_clean.drop(columns=['Fraud_Label'])
y = df_clean['Fraud_Label']

print(f"Features (X): {X.shape[1]} columns")
print(X.columns.tolist())
print(f"\nTarget (y): {y.shape[0]:,} rows")
print(f"  Fraud rate: {y.mean()*100:.2f}%")


In [ ]:
# ── 4.5  Train / Test Split ──────────────────────────────────────────────────
# We split data: 80% for training the model, 20% for testing it.
# stratify=y ensures both sets have the same fraud percentage.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% goes to testing
    random_state=42,      # fixed seed = reproducible results
    stratify=y            # keep fraud ratio the same in both sets
)

print(f"Training set  : {X_train.shape[0]:,} rows")
print(f"Testing  set  : {X_test.shape[0]:,} rows")
print(f"Fraud in train: {y_train.mean()*100:.2f}%")
print(f"Fraud in test : {y_test.mean()*100:.2f}%")


In [ ]:
# ── 4.6  SMOTE — Fix Class Imbalance ─────────────────────────────────────────
# The dataset is imbalanced (very few frauds vs many legitimate transactions).
# SMOTE creates SYNTHETIC fraud samples so the model sees equal amounts of both.
# IMPORTANT: Apply SMOTE ONLY on training data, NEVER on test data.

print(f"Before SMOTE:")
print(f"  Legitimate: {(y_train==0).sum():,}")
print(f"  Fraud:      {(y_train==1).sum():,}")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Legitimate: {(y_train_sm==0).sum():,}")
print(f"  Fraud:      {(y_train_sm==1).sum():,}")
print(f"✅ Dataset is now balanced!")


In [ ]:
# ── 4.7  Feature Scaling ──────────────────────────────────────────────────────
# Scaling puts all numbers on the same range.
# Why? Without scaling, "Account_Balance" (thousands) would dominate "Card_Age" (days).
# StandardScaler: transforms to mean=0, std=1

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)   # fit + transform on training data
X_test_sc  = scaler.transform(X_test)           # ONLY transform on test data (no fitting)

print("✅ Features scaled!")
print(f"   Mean of first feature (approx): {X_train_sc[:, 0].mean():.4f}  (should be ~0)")
print(f"   Std  of first feature (approx): {X_train_sc[:, 0].std():.4f}   (should be ~1)")


---
## 🤖 Section 5 — Model Training
We train **6 different models** and compare them at the end.
Each model takes a different approach to detecting fraud.

| Model | Type | Idea |
|---|---|---|
| Logistic Regression | Supervised | Draws a straight decision boundary |
| Decision Tree | Supervised | Asks yes/no questions to classify |
| Random Forest | Supervised | Many decision trees voting together |
| XGBoost | Supervised | Trees that learn from previous mistakes |
| Isolation Forest | Unsupervised | Isolates outliers (fraud = easy to isolate) |
| Voting Ensemble | Combined | Combines LR + RF + XGBoost for best result |


In [ ]:
# ── Helper function: evaluate any model and return its metrics ────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, needs_scale=True):
    """Train a model and return a dictionary of metrics."""
    Xtr = X_tr if needs_scale else X_train_sm
    Xte = X_te if needs_scale else X_test

    model.fit(Xtr, y_tr)
    y_pred = model.predict(Xte)

    # Get probability scores for AUC-ROC (some models use predict_proba)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(Xte)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_prob = model.decision_function(Xte)
    else:
        y_prob = y_pred

    metrics = {
        'Model'     : name,
        'Accuracy'  : round(accuracy_score(y_te, y_pred) * 100, 2),
        'Precision' : round(precision_score(y_te, y_pred, zero_division=0) * 100, 2),
        'Recall'    : round(recall_score(y_te, y_pred, zero_division=0) * 100, 2),
        'F1 Score'  : round(f1_score(y_te, y_pred, zero_division=0) * 100, 2),
        'AUC-ROC'   : round(roc_auc_score(y_te, y_prob) * 100, 2),
        '_model'    : model,
        '_y_pred'   : y_pred,
        '_y_prob'   : y_prob,
    }
    print(f"✅ {name:30s} | Accuracy: {metrics['Accuracy']}%  F1: {metrics['F1 Score']}%  AUC: {metrics['AUC-ROC']}%")
    return metrics

all_results = []
print("Training models...\n")


In [ ]:
# ── Model 1: Logistic Regression ─────────────────────────────────────────────
# Simplest model. Finds a linear boundary between fraud and legitimate.
# Good baseline — if more complex models don't beat this, something is wrong.

lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
result_lr = evaluate_model('Logistic Regression', lr,
                            X_train_sc, y_train_sm, X_test_sc, y_test)
all_results.append(result_lr)


In [ ]:
# ── Model 2: Decision Tree ───────────────────────────────────────────────────
# Makes a tree of if/else questions to classify each transaction.
# max_depth=10 prevents overfitting (memorising training data).

dt = DecisionTreeClassifier(max_depth=10, random_state=42)
result_dt = evaluate_model('Decision Tree', dt,
                            X_train_sc, y_train_sm, X_test_sc, y_test)
all_results.append(result_dt)


In [ ]:
# ── Model 3: Random Forest ───────────────────────────────────────────────────
# Builds 100 decision trees and takes a majority vote.
# More robust than a single tree — less likely to overfit.
# n_jobs=-1 uses all CPU cores (faster training).

rf = RandomForestClassifier(n_estimators=100, max_depth=15,
                             random_state=42, n_jobs=-1)
result_rf = evaluate_model('Random Forest', rf,
                            X_train_sc, y_train_sm, X_test_sc, y_test)
all_results.append(result_rf)


In [ ]:
# ── Model 4: XGBoost ─────────────────────────────────────────────────────────
# "Extreme Gradient Boosting": builds trees sequentially,
# each one correcting the mistakes of the previous one.
# Often the best performer on tabular (table) data.

xgb = XGBClassifier(n_estimators=200, learning_rate=0.1,
                     max_depth=6, random_state=42,
                     use_label_encoder=False, eval_metric='logloss',
                     verbosity=0, n_jobs=-1)
result_xgb = evaluate_model('XGBoost', xgb,
                              X_train_sc, y_train_sm, X_test_sc, y_test)
all_results.append(result_xgb)


In [ ]:
# ── Model 5: Isolation Forest (Anomaly Detection) ────────────────────────────
# UNSUPERVISED model — it doesn't look at the fraud labels during training.
# Idea: fraud transactions are "anomalies" (unusual), so they're easier to isolate.
# contamination = expected % of fraud in the data.

iso = IsolationForest(n_estimators=100, contamination=0.1,
                      random_state=42, n_jobs=-1)

# Isolation Forest uses the unscaled SMOTE data (unsupervised, no labels)
iso.fit(X_train_sm, y_train_sm)
iso_raw = iso.predict(X_test)                 # returns -1 (anomaly) or +1 (normal)
iso_pred = np.where(iso_raw == -1, 1, 0)      # convert: -1 → 1 (fraud), +1 → 0 (legit)
iso_score = -iso.decision_function(X_test)    # higher score = more anomalous

iso_metrics = {
    'Model'     : 'Isolation Forest',
    'Accuracy'  : round(accuracy_score(y_test, iso_pred) * 100, 2),
    'Precision' : round(precision_score(y_test, iso_pred, zero_division=0) * 100, 2),
    'Recall'    : round(recall_score(y_test, iso_pred, zero_division=0) * 100, 2),
    'F1 Score'  : round(f1_score(y_test, iso_pred, zero_division=0) * 100, 2),
    'AUC-ROC'   : round(roc_auc_score(y_test, iso_score) * 100, 2),
    '_model'    : iso, '_y_pred': iso_pred, '_y_prob': iso_score,
}
all_results.append(iso_metrics)
print(f"✅ {'Isolation Forest':30s} | Accuracy: {iso_metrics['Accuracy']}%  "
      f"F1: {iso_metrics['F1 Score']}%  AUC: {iso_metrics['AUC-ROC']}%")


In [ ]:
# ── Model 6: Voting Ensemble ─────────────────────────────────────────────────
# Combines Logistic Regression + Random Forest + XGBoost.
# "Soft" voting = averages the probability predictions (better than hard voting).
# Usually beats any single model by reducing individual errors.

ensemble = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(C=1.0, max_iter=500, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)),
        ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
                               random_state=42, use_label_encoder=False,
                               eval_metric='logloss', verbosity=0, n_jobs=-1)),
    ],
    voting='soft'   # uses probability averages
)

result_ens = evaluate_model('Voting Ensemble (LR+RF+XGB)', ensemble,
                             X_train_sc, y_train_sm, X_test_sc, y_test)
all_results.append(result_ens)
print("\n✅ All models trained!")


---
## 📈 Section 6 — Evaluation & Comparison

Now we compare all models on the same test set and pick the best one.

**Key metrics explained:**
- **Accuracy**: % of total predictions that were correct
- **Precision**: Of all transactions flagged as fraud, how many actually were? (avoid false alarms)
- **Recall**: Of all actual frauds, how many did we catch? (avoid missing fraud)
- **F1 Score**: Balance between Precision and Recall — *our main metric for fraud*
- **AUC-ROC**: How well the model separates fraud from legitimate (1.0 = perfect)


In [ ]:
# ── 6.1  Model Comparison Table ──────────────────────────────────────────────
metrics_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                            for r in all_results])
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

# Add rank column
results_df.insert(0, 'Rank', range(1, len(results_df)+1))

# Highlight best row
print("=" * 70)
print("MODEL COMPARISON — sorted by F1 Score (best first)")
print("=" * 70)
print(results_df.to_string(index=False))
print("=" * 70)
print(f"\n🏆 Best model: {results_df.iloc[0]['Model']}  (F1 = {results_df.iloc[0]['F1 Score']}%)")

# Save to CSV for use in the Streamlit dashboard
results_df.to_csv('model_comparison.csv', index=False)
print("💾 Saved as model_comparison.csv")


In [ ]:
# ── 6.2  Visual Model Comparison Bar Chart ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(results_df))
width = 0.15
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * width, results_df[metric], width,
                  label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score (%)')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(loc='upper right')
ax.axhline(y=90, color='gray', linestyle='--', alpha=0.4, label='90% line')

plt.tight_layout()
plt.savefig('model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6.3  Confusion Matrices for top 3 models ─────────────────────────────────
# Confusion Matrix shows:
#   True Negative  (correctly said NOT fraud) | False Positive (wrongly said fraud)
#   False Negative (missed real fraud)         | True Positive  (correctly caught fraud)

top3_names = results_df['Model'].head(3).tolist()
top3_results = [r for r in all_results if r['Model'] in top3_names]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, result in zip(axes, top3_results):
    cm = confusion_matrix(y_test, result['_y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'])
    ax.set_title(f"{result['Model']}\nF1={result['F1 Score']}%", fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6.4  ROC Curves for all models ───────────────────────────────────────────
# ROC Curve plots True Positive Rate vs False Positive Rate.
# The more the curve bends toward the top-left, the better the model.
# AUC = area under the curve (1.0 = perfect, 0.5 = random guessing)

plt.figure(figsize=(10, 7))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#00BCD4']
for result, color in zip(all_results, colors):
    fpr, tpr, _ = roc_curve(y_test, result['_y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{result['Model']} (AUC={result['AUC-ROC']}%)")

plt.plot([0,1], [0,1], 'k--', lw=1.5, label='Random Classifier (AUC=50%)')
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6.5  Feature Importance (from Random Forest) ─────────────────────────────
# Random Forest tells us which features matter most for detecting fraud.
# This is valuable insight: which transaction details are the biggest red flags?

rf_model = result_rf['_model']
feature_names = X.columns.tolist()
importances = pd.Series(rf_model.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importances.plot(kind='barh', color='#2196F3', edgecolor='white')
plt.title('Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6.6  Detailed Classification Report for Best Model ───────────────────────
best_result = [r for r in all_results if r['Model'] == results_df.iloc[0]['Model']][0]

print(f"Detailed Report — {best_result['Model']}")
print("=" * 55)
print(classification_report(y_test, best_result['_y_pred'],
                              target_names=['Legitimate', 'Fraud']))


---
## 💾 Section 7 — Save Models & Preprocessing Objects
We save the best model and all preprocessing objects to disk.
The Streamlit dashboard will load these files to make live predictions.

> ⚠️ **Important:** Always save the scaler and encoders too — not just the model!
> When the dashboard receives a new transaction, it must transform it the same way
> we transformed the training data.


In [ ]:
import os
os.makedirs('models', exist_ok=True)   # create models/ folder if it doesn't exist

# ── Save the best model ───────────────────────────────────────────────────────
best_model_obj = best_result['_model']
joblib.dump(best_model_obj, 'models/best_fraud_model.pkl')
print(f"✅ Saved best model: {best_result['Model']} → models/best_fraud_model.pkl")

# ── Save the Voting Ensemble specifically (for dashboard) ─────────────────────
ens_result = [r for r in all_results if r['Model'] == 'Voting Ensemble (LR+RF+XGB)']
if ens_result:
    joblib.dump(ens_result[0]['_model'], 'models/ensemble_model.pkl')
    print("✅ Saved ensemble model → models/ensemble_model.pkl")

# ── Save the scaler ───────────────────────────────────────────────────────────
joblib.dump(scaler, 'models/scaler.pkl')
print("✅ Saved scaler → models/scaler.pkl")

# ── Save the label encoders ───────────────────────────────────────────────────
joblib.dump(label_encoders, 'models/label_encoders.pkl')
print("✅ Saved label encoders → models/label_encoders.pkl")

# ── Save the feature column list ─────────────────────────────────────────────
joblib.dump(X.columns.tolist(), 'models/feature_columns.pkl')
print("✅ Saved feature columns → models/feature_columns.pkl")

print("\n📁 All files saved in models/ folder:")
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}')
    print(f"   {f:35s} {size/1024:.1f} KB")


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  PROJECT 1 — FINANCIAL FRAUD DETECTION COMPLETE! 🎉")
print("=" * 60)
print()
print("Files generated:")
print("  📊 eda_class_distribution.png")
print("  📊 eda_transaction_amount.png")
print("  📊 eda_fraud_by_type.png")
print("  📊 eda_fraud_by_device_location.png")
print("  📊 eda_boxplots.png")
print("  📊 model_comparison_chart.png")
print("  📊 confusion_matrices.png")
print("  📊 roc_curves.png")
print("  📊 feature_importance.png")
print("  📄 model_comparison.csv")
print("  🤖 models/best_fraud_model.pkl")
print("  🤖 models/ensemble_model.pkl")
print("  🤖 models/scaler.pkl")
print("  🤖 models/label_encoders.pkl")
print()
print(f"Best Model : {results_df.iloc[0]['Model']}")
print(f"F1 Score   : {results_df.iloc[0]['F1 Score']}%")
print(f"AUC-ROC    : {results_df.iloc[0]['AUC-ROC']}%")
print()
print("NEXT STEP: Open and run streamlit_app.py for the dashboard!")
